# 01 — Dataset Scanning, Multi-View Trial Alignment & WitMotion IMU Loading

This notebook demonstrates **Stage 1** of the `monocular-3d-gait-tracking-deltav2` (**Gait360**) pipeline:
1. Resolving the project root and validating dataset paths via `src.utils.path_validator`.
2. Loading the multi-view trial alignment map (`data/cached_dataset_map.json`) via `DatasetScanner`.
3. Loading and filtering WitMotion IMU ground-truth thigh kinematics (`src/analysis/gt_loader.py` and `src/preprocessing/signals/filtering.py`).

In [ ]:
import sys
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'examples' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.path_validator import get_project_root, resolve_config_path
from src.preprocessing.signals.filtering import ButterworthFilter
from src.preprocessing.signals.frequency_analysis import FrequencyAnalyzer
print('Project Root:', get_project_root())
print('Default Config:', resolve_config_path())

In [ ]:
# Inspect the stratified 40-video benchmark manifest
import csv
manifest_40 = PROJECT_ROOT / 'data/selected_inference_videos_40.csv'
with manifest_40.open('r', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))
print(f'Total benchmark videos: {len(rows)}')
for r in rows[:5]:
    print(f"  {r['video_name']:22s} | Category: {r['category']} | Subject: {r['person_id']}")

In [ ]:
# Demonstrate zero-phase Butterworth filtering and FFT step cadence estimation on IMU pitch
fs = 50.0  # WitMotion IMU sampling rate (Hz)
t = np.linspace(0, 4.0, int(4.0 * fs), endpoint=False)
raw_pitch = 22.0 * np.sin(2 * np.pi * 0.95 * t) + 3.5 * np.random.RandomState(42).randn(len(t))
bw_filter = ButterworthFilter(order=4)
clean_pitch = bw_filter.lowpass(raw_pitch, cutoff=6.0, fs=fs)
freq_hz = FrequencyAnalyzer().estimate_dominant_frequency(clean_pitch, fs=fs)
print(f'Estimated stride frequency: {freq_hz:.2f} Hz ({freq_hz * 60:.1f} strides/min)')